# Bước 05/06 — Train YOLO11n@640, 50 epoch (data ablation)

**Data ablation, KHÔNG phải benchmark V6.5 vs V6.4 cuối.** V6.2E TRAIN + MPCD/PowerEquipment TRAIN đã KEEP + synthetic đã duyệt; đánh giá trên **VAL V6.2E đóng băng**; TEST không bao giờ được tạo. `yolo11n.pt`, 640, 50 ep, batch -1, seed 20261007. Cần GPU CUDA (hoặc dùng launcher Kaggle).

Experiment ID: `v6p5-abl01-yolo11n-640-e50` · checkpoint sync lên Camber mỗi 5 epoch · mất máy/kernel → chạy lại là resume từ `last.pt`.

| | Camber Stash |
|---|---|
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa/v6p5_compose_<12hex>/ (COMPOSE)` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p2e/v6p2e_23562a45b343/ + archive base/overlay` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/mpcd/mpcd_broken_strand_stage.tar.gz` |
| **Input** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/power_equipment/power_equipment_stage.tar.gz` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation/v6p5_y11n_ablation_<12hex>/{last.pt, best.pt, run_state.json, results.csv, args.yaml}` |
| **Output** | `(local) runs/yolo11n_640/v6p5-abl01-yolo11n-640-e50__v6p5_compose_<12hex>/` |

Chạy lại notebook bất cứ lúc nào: bước đã xong + đúng config sẽ được **reuse**, bước dở dang sẽ **resume**.
Bước tiếp theo: **06_evaluate_vs_v6p2e_baseline**

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
# ===== DATASET TRÊN CAMBER — điền tag nếu muốn chọn tay, None = tự động =====
PARENT_DATASET = "v6p2e_23562a45b343"   # V6.2E: cố định, KHÔNG đổi
EXTERNAL_STAGE = None   # vd "v6p5_ext_0123456789ab"     (tạo ở bước 03)
COMPOSE        = None   # vd "v6p5_compose_0123456789ab" (tạo ở bước 04)
REQUIRE_CAMBER = True   # lưu chính trên Camber -> dừng nếu chưa đăng nhập

from yolov11sdi.camber import CamberClient
from yolov11sdi.environment import load_camber_api_key
from yolov11sdi.reporting import status
cfg = pipeline.load(stage_tag=EXTERNAL_STAGE, compose_tag=COMPOSE, parent_tag=PARENT_DATASET)
load_camber_api_key(paths.env)
cam = CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))
if REQUIRE_CAMBER and not cam.available():
    raise RuntimeError("Camber chưa sẵn sàng: chạy `camber login` hoặc export CAMBER_API_KEY "
                       "(CAMBER_BIN=... nếu CLI không nằm trong PATH). Chỉ chạy local: REQUIRE_CAMBER=False")
display(status.dataset_card(cfg, paths, cam))

In [ ]:
pipeline.run_stage("yolo11n_ablation", cfg, dry_run=True)

In [ ]:
outcome = pipeline.run_stage("yolo11n_ablation", cfg, force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
# Kiểm tra output đã thật sự nằm trên Camber chưa
display(pipeline.verify_on_camber("v6p5-abl01-yolo11n-640-e50", cfg))

In [ ]:
from yolov11sdi.reporting.status import latest_checkpoint
print(latest_checkpoint(paths))
runs = sorted(paths.runs.glob("yolo11n_640/*/results.csv"))
if runs:
    r = pd.read_csv(runs[-1]); r.columns = [c.strip() for c in r.columns]
    display(r.tail())
    r.plot(x="epoch", y=[c for c in r.columns if "mAP50" in c], figsize=(8, 4))